<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/NDFI_LNDT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Installing Python Libraries

YouTo install Python libraries in Google Colab, you use the `pip install` command.  You can also use a `!` before the command to execute it as a shell command directly within a notebook cell.

In [ ]:
# Example: Install the 'requests' library
!pip install requests

print("Library 'requests' installed successfully!")

In [ ]:
import ee

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-jaalvalcan')


In [ ]:
import geemap

In [ ]:

# ==============================================================================
# 2. DEFINE YOUR EXACT COORDINATES (AOI)
# ==============================================================================
aoi_coordinates = [
    [-64.566736, -10.292288],
    [-64.566736, -10.25234],
    [-64.479618, -10.25234],
    [-64.479618, -10.292288],
    [-64.566736, -10.292288]
]

aoi = ee.Geometry.Polygon(coords=[aoi_coordinates], proj='EPSG:4326', geodesic=False)

# ==============================================================================
# 3. FETCH AND CLEAN LANDSAT 8 IMAGERY
# ==============================================================================
# Load Landsat 8 Surface Reflectance Tier 1 data
l8_collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                 .filterBounds(aoi)
                 # Focus on the Amazonian dry season window for maximum canopy distinction
                 .filterDate('2024-05-01', '2024-10-31')
                 .sort('CLOUD_COVER')
                 .filter(ee.Filter.lt('CLOUD_COVER', 10)))

raw_l8_image = l8_collection.first()

# Apply the official USGS Landsat 8 Level-2 scaling factors for Surface Reflectance
def scale_l8(img):
    optical_bands = img.select('SR_B[2-7]').multiply(0.0000275).add(-0.2)
    return img.addBands(optical_bands, None, True)

scaled_l8_image = scale_l8(raw_l8_image)

# No spatial resampling is needed here, as Landsat bands 2-7 natively share a 30m grid

# ==============================================================================
# 4. CALCULATE TRADITIONAL NDVI
# ==============================================================================
# Landsat 8 NDVI = (NIR - Red) / (NIR + Red) -> (SR_B5 - SR_B4) / (SR_B5 + SR_B4)
ndvi = scaled_l8_image.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')

# ==============================================================================
# 5. LINEAR SPECTRAL UNMIXING FOR LANDSAT 8 NDFI
# ==============================================================================
# Classic Souza Amazonian endmember parameters calibrated for Landsat 8:
# Order matching bands: ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
gv = [0.05, 0.08, 0.06, 0.62, 0.28, 0.12]    # Green Vegetation
npv = [0.12, 0.18, 0.24, 0.45, 0.68, 0.42]   # Non-Photosynthetic Vegetation (Wood/Slash)
soil = [0.20, 0.32, 0.44, 0.52, 0.72, 0.58]  # Exposed Soil
shade = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01] # Forest Shade

endmembers = [gv, npv, soil, shade]
bands_to_unmix = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']

# Run the linear unmixing matrix
fractions = scaled_l8_image.select(bands_to_unmix).unmix(endmembers)
fractions = fractions.rename(['GV', 'NPV', 'Soil', 'Shade']).clamp(0, 1)

# ==============================================================================
# 6. CALCULATE NDFI FORMULA
# ==============================================================================
gv_fraction = fractions.select('GV')
shade_fraction = fractions.select('Shade')
npv_soil = fractions.select('NPV').add(fractions.select('Soil'))

# Canopy shade normalization step
gv_shade = gv_fraction.divide(ee.Image(1.0).subtract(shade_fraction))

# Execute the NDFI algebraic equation
denominator = gv_shade.add(npv_soil).add(0.0001)
ndfi = (gv_shade.subtract(npv_soil)).divide(denominator).rename('NDFI')

# ==============================================================================
# 7. INTERACTIVE VISUALIZATION & LEGENDS
# ==============================================================================
Map = geemap.Map()
Map.centerObject(aoi, zoom=14)

# Visualization Parameters adjusted for Landsat 8
rgb_vis = {'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 'min': 0, 'max': 0.15, 'gamma': 1.2}
ndvi_vis = {'min': 0.2, 'max': 0.9, 'palette': ['#e5f5f9', '#99d8c9', '#2ca25f']}
ndfi_colors = ['#d7191c', '#fdae61', '#ffffbf', '#a6d96a', '#1a9641']
ndfi_vis = {'min': -0.5, 'max': 1.0, 'palette': ndfi_colors}

# Display layers to interactive viewport
Map.add_layer(scaled_l8_image.clip(aoi), rgb_vis, '1. Landsat 8 True Color (RGB)')
Map.add_layer(ndvi.clip(aoi), ndvi_vis, '2. Traditional NDVI (Greenness)')
Map.add_layer(ndfi.clip(aoi), ndfi_vis, '3. VM0047 NDFI (Structural Carbon)')

# Render the colorbar gradient
Map.add_colorbar(
    vis_params=ndfi_vis,
    label="Landsat NDFI Structural Value",
    orientation="horizontal",
    layer_name="NDFI Colorbar"
)

# Render the categorical interpretation legend
legend_dict = {
    'Intact Canopy Forest (0.75 to 1.0)': '#1a9641',
    'Secondary Growth / Regeneration (0.25 to 0.75)': '#a6d96a',
    'Canopy Gaps / Logging Interventions (0.0 to 0.25)': '#ffffbf',
    'Exposed Branches / Slash Accumulation (-2.5 to 0.0)': '#fdae61',
    'Bare Ground / Farm Clearances (-0.5 to -0.25)': '#d7191c'
}

Map.add_legend(title="VM0047 NDFI Interpretation", legend_dict=legend_dict)

display(Map)